# Experiment 10: Transfer Learning Using Pre-trained VGG16 on an Image Dataset

**Objective:** Implement transfer learning using the pre-trained VGG16 model on an image dataset.

**Language:** Python  
**Estimated Time:** 2 hours  
**Libraries Used:** NumPy, Matplotlib, TensorFlow / Keras, scikit-learn


## 1. Introduction

Transfer learning is a technique where a model trained on a large dataset is reused for a different but related task. In this experiment, we use **VGG16**, a popular convolutional neural network pre-trained on the **ImageNet** dataset.

Instead of training a deep network from scratch, we use VGG16 as a feature extractor and add our own classification layers on top.


## 2. Aim of the Experiment

In this notebook, we will:

1. Load an image dataset from folders
2. Preprocess the images for VGG16
3. Load the pre-trained VGG16 model
4. Freeze base layers
5. Add custom dense layers for classification
6. Train and evaluate the model
7. Visualize predictions


In [ ]:
# Install if needed
# !pip install tensorflow matplotlib numpy scikit-learn

## 3. Import Libraries


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix

np.random.seed(42)
tf.random.set_seed(42)
print('TensorFlow Version:', tf.__version__)

## 4. Dataset Structure

Keep your dataset in the following folder structure:

```text
dataset/
    train/
        class_1/
        class_2/
        ...
    test/
        class_1/
        class_2/
        ...
```

Each class folder should contain images belonging to that class.


In [ ]:
# Update these paths according to your dataset location
train_dir = 'dataset/train'
test_dir = 'dataset/test'

img_height, img_width = 224, 224
batch_size = 32

## 5. Data Preprocessing

VGG16 expects input images of size **224 x 224** and uses a specific preprocessing function.


In [ ]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

val_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

class_names = list(train_generator.class_indices.keys())
num_classes = len(class_names)

print('Classes:', class_names)
print('Number of classes:', num_classes)

## 6. Visualize Sample Images


In [ ]:
images, labels = next(train_generator)

plt.figure(figsize=(10, 6))
for i in range(min(8, len(images))):
    plt.subplot(2, 4, i + 1)
    img = images[i].copy()
    img = img - img.min()
    img = img / (img.max() + 1e-8)
    plt.imshow(img)
    plt.title(class_names[np.argmax(labels[i])])
    plt.axis('off')
plt.tight_layout()
plt.show()

## 7. Load Pre-trained VGG16 Model

We load VGG16 without its top classification layers and use its convolutional base as a feature extractor.


In [ ]:
base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(img_height, img_width, 3)
)

# Freeze all convolutional layers
for layer in base_model.layers:
    layer.trainable = False

base_model.summary()

## 8. Build the Transfer Learning Model


In [ ]:
model = Sequential([
    base_model,
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 9. Train the Model


In [ ]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    verbose=1
)

## 10. Plot Accuracy and Loss Curves


In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## 11. Evaluate on Test Dataset


In [ ]:
test_loss, test_accuracy = model.evaluate(test_generator, verbose=0)
print('Test Loss:', round(test_loss, 4))
print('Test Accuracy:', round(test_accuracy, 4))

## 12. Predictions and Classification Report


In [ ]:
pred_probs = model.predict(test_generator, verbose=0)
pred_labels = np.argmax(pred_probs, axis=1)
true_labels = test_generator.classes

print('Classification Report:\n')
print(classification_report(true_labels, pred_labels, target_names=class_names))

In [ ]:
cm = confusion_matrix(true_labels, pred_labels)
print('Confusion Matrix:\n')
print(cm)

## 13. Visualize Predictions


In [ ]:
test_images, test_labels = next(test_generator)
pred_batch = model.predict(test_images, verbose=0)

plt.figure(figsize=(12, 6))
for i in range(min(8, len(test_images))):
    plt.subplot(2, 4, i + 1)
    img = test_images[i].copy()
    img = img - img.min()
    img = img / (img.max() + 1e-8)
    plt.imshow(img)
    true_name = class_names[np.argmax(test_labels[i])]
    pred_name = class_names[np.argmax(pred_batch[i])]
    plt.title(f'True: {true_name}\nPred: {pred_name}')
    plt.axis('off')
plt.tight_layout()
plt.show()

## 14. Optional Fine-Tuning

After initial training, you can unfreeze some deeper layers of VGG16 and train with a very small learning rate to improve performance further.


In [ ]:
# Example fine-tuning setup (optional)
# for layer in base_model.layers[-4:]:
#     layer.trainable = True
#
# model.compile(
#     optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
#     loss='categorical_crossentropy',
#     metrics=['accuracy']
# )
#
# history_finetune = model.fit(
#     train_generator,
#     validation_data=val_generator,
#     epochs=5,
#     verbose=1
# )

## 15. Conclusion

In this experiment, we implemented transfer learning using the pre-trained VGG16 model. The convolutional base extracted useful image features, and the custom classifier learned to classify the dataset with fewer training resources than training a deep model from scratch.


## 16. Viva Questions

1. What is transfer learning?
2. Why do we use a pre-trained model like VGG16?
3. Why are the base model layers frozen initially?
4. What is the difference between feature extraction and fine-tuning?
5. Why does VGG16 require 224x224 input images?
